# Workshop introductie pydov

## Wat is pydov?

- Een Python package om gemakkelijk DOV data te kunnen gebruiken in andere scripts en tools
  - Zoeken op attributen en locatie​
  - Combineren met andere datasets
  - Resultaten beschikbaar in een Pandas DataFrame​

- Een referentie client implementatie van onze metadata, WFS en XML services

- Een community project
  - Open ontwikkeling op GitHub:​ https://github.com/DOV-Vlaanderen/pydov/
  - Open-source licentie: MIT

- Zelf bijdragen?
  - Issues voor vragen
  - Documentatie
  - Code

## Quick start

In [1]:
from pydov.search.boring import BoringSearch

from pydov.util.location import Within, Box

from owslib.fes2 import PropertyIsGreaterThan

boring_search = BoringSearch()

dataframe = boring_search.search(
    query=PropertyIsGreaterThan(propertyname='diepte_tot_m', literal='550'),
    location=Within(Box(107500, 202000, 108500, 203000, epsg=31370))
)

dataframe

[000/001] .
[000/002] cc


,pkey_boring,boornummer,x,y,mv_mtaw,start_boring_mtaw,gemeente,diepte_boring_van,diepte_boring_tot,datum_aanvang,uitvoerder,boorgatmeting,diepte_methode_van,diepte_methode_tot,boormethode
0,https://www.dov.vlaanderen.be/data/boring/1989...,kb14d40e-B777,108015.0,202860.0,5.0,5.0,Gent,0.0,660.0,1989-01-25,onbekend,False,0.0,660.0,onbekend
1,https://www.dov.vlaanderen.be/data/boring/1972...,kb14d40e-B778,108090.0,202835.0,5.0,5.0,Gent,0.0,600.0,1972-05-17,onbekend,False,0.0,600.0,onbekend


## Datasets selecteren
> Meer info: https://pydov.readthedocs.io/en/stable/select_datasets.html

Om data op te halen moet je eerst een keuze maken welke datasets je wil bevragen.

### Zoekobjecten

Elk van de beschikbare datasets heeft een bijhorend zoekobject waarmee je de data kan bevragen.

Volgende code maakt drie zoekobjecten aan, om respectievelijk data over Boringen, Monsters en Observaties te kunnen bevragen.

In [4]:
from pydov.search.boring import BoringSearch
from pydov.search.monster import MonsterSearch
from pydov.search.observatie import ObservatieSearch

boring_search = BoringSearch()
monster_search = MonsterSearch()
observatie_search = ObservatieSearch()

### Object types
> Meer info: https://pydov.readthedocs.io/en/stable/output_fields.html#customizing-object-types-and-subtypes

Elk zoekobject is gekoppeld aan een object type, dat bepaalt welke velden er standaard in het resultaat beschikbaar zijn.

Bij het aanmaken van het zoekobject kan je optioneel het objecttype opgeven, om zo te kunnen beïnvloeden welke velden er teruggegeven zullen worden. Volgende code is equivalent aan de eerdere versie van `observatie_search`:

In [5]:
from pydov.search.observatie import ObservatieSearch

from pydov.types.observatie import Observatie

observatie_search = ObservatieSearch(
    objecttype=Observatie
)

Volgend schema verduidelijkt het verschil tussen zoekobjecten, objecttypes en subtypes. Een zoekobject bepaalt op welke velden er gezocht kan worden, een objecttype bepaalt welke velden er in het resultaat teruggegeven kunnen worden. Per record uit het zoekresultaat is er maximum één resultaat uit het (hoofd) objecttype, en kunnen er meerdere records uit het subtype zijn.

![objecttypes](../objecttypes.svg)

### Fieldsets

Bij sommige objecttypes zijn er extra velden beschikbaar die niet standaard aanwezig zijn in het resultaat, maar die eenvoudig toegevoegd kunnen worden. Via de methode `get_fieldsets()` bij een objecttype kan je opvragen welke sets beschikbaar zijn.

In [6]:
from pydov.search.observatie import ObservatieSearch
from pydov.types.observatie import Observatie

Observatie.get_fieldsets()

{'ObservatieDetails': {'name': 'ObservatieDetails',
  'class': pydov.types.observatie.ObservatieDetails,
  'definition': 'Fieldset containing fields with extra details about the observation. It has the following fields: betrouwbaarheid, geobserveerd_object_type, geobserveerd_object_naam, geobserveerd_object_permkey.'}}

In [7]:
from pydov.types.observatie import ObservatieDetails

observatie_search = ObservatieSearch(
    objecttype=Observatie.with_extra_fields(ObservatieDetails)
)

### Subtypes

Bij sommige objecttypes zijn er extra subtypes beschikbaar, die gebruikt kunnen worden in plaats van het standaard subtype. Via de methode `get_subtypes()` bij een objecttype kan je opvragen welke subtypes beschikbaar zijn.

In [8]:
from pydov.search.observatie import ObservatieSearch
from pydov.types.observatie import Observatie

Observatie.get_subtypes()

{'Fractiemeting': {'name': 'Fractiemeting',
  'class': pydov.types.observatie.Fractiemeting,
  'definition': 'Subtype showing the details of a fraction measurement. It has the following fields: fractiemeting_ondergrens, fractiemeting_bovengrens, fractiemeting_waarde.'},
 'Meetreeks': {'name': 'Meetreeks',
  'class': pydov.types.observatie.Meetreeks,
  'definition': 'Subtype showing the details of a measurement series. It has the following fields: meetreeks_meetpunt_parameter, meetreeks_meetpunt, meetreeks_meetpunt_eenheid, meetreeks_meetwaarde_parameter, meetreeks_meetwaarde, meetreeks_meetwaarde_eenheid.'},
 'ObservatieHerhaling': {'name': 'ObservatieHerhaling',
  'class': pydov.types.observatie.ObservatieHerhaling,
  'definition': 'Subtype showing the repetition information of an observation. It has the following fields: herhaling_aantal, herhaling_minimum, herhaling_maximum, herhaling_standaardafwijking.'},
 'SecundaireParameter': {'name': 'SecundaireParameter',
  'class': pydov.typ

In [9]:
from pydov.types.observatie import SecundaireParameter

observatie_search = ObservatieSearch(
    objecttype=Observatie.with_subtype(SecundaireParameter)
)

## Zoeken op locatie
> Meer info: https://pydov.readthedocs.io/en/stable/query_location.html

Geografisch zoeken kan met de `location` parameter van de `search` methode. Je geeft hieraan een geografische filter en een geometrie-object mee, ofwel een filter factory op basis van een geodataframe of vector GIS bestand.

### Overlap met rechthoek

Zoeken naar objecten die overlappen met een rechthoekig gebied is eenvoudig:


In [10]:
from pydov.search.observatie import ObservatieSearch
from pydov.util.location import Within, Box

observatie_search = ObservatieSearch()

observatie_search.search(
    location=Within(Box(minx=200000, miny=211000, maxx=201000, maxy=212000, epsg=31370))
)

[000/001] .


,pkey_observatie,pkey_parent,fenomeentijd,diepte_van_m,diepte_tot_m,parametergroep,parameter,detectieconditie,resultaat,eenheid,methode,uitvoerder,herkomst
0,https://www.dov.vlaanderen.be/data/observatie/...,https://www.dov.vlaanderen.be/data/monster/196...,1968-01-01,NaN,NaN,Bodem_chemisch,pH H2O (ph_h2o),NaN,5.8,-,NaN,Centrum voor Grondonderzoek (C.V.G.),LABO
1,https://www.dov.vlaanderen.be/data/observatie/...,https://www.dov.vlaanderen.be/data/monster/196...,1968-01-01,NaN,NaN,Bodem_chemisch,Organische C - percentage (organische_c_perc),NaN,3.69,%,Aardewerk nieuwe methode organische koolstof,Centrum voor Grondonderzoek (C.V.G.),LABO
2,https://www.dov.vlaanderen.be/data/observatie/...,https://www.dov.vlaanderen.be/data/monster/196...,1968-01-01,NaN,NaN,Bodem_fysisch_textuur,Textuur - grove fractie (groter dan 2000 µm) (...,NaN,0.0,%,NaN,Centrum voor Grondonderzoek (C.V.G.),LABO
3,https://www.dov.vlaanderen.be/data/observatie/...,https://www.dov.vlaanderen.be/data/monster/196...,1968-01-01,NaN,NaN,Bodem_fysisch_textuur,Textuur - handmatig - fout groter dan 5% (tex...,NaN,ja,NaN,NaN,Centrum voor Grondonderzoek (C.V.G.),LABO
4,https://www.dov.vlaanderen.be/data/observatie/...,https://www.dov.vlaanderen.be/data/monster/196...,1968-01-01,NaN,NaN,Bodem_fysisch_textuur,Mediaan van de textuurfracties (textuur_mediaan),NaN,140,µm,NaN,Centrum voor Grondonderzoek (C.V.G.),LABO
5,https://www.dov.vlaanderen.be/data/observatie/...,https://www.dov.vlaanderen.be/data/monster/196...,1968-01-01,NaN,NaN,Bodem_chemisch,Calciumcarbonaatgehalte (caco3_gehalte),NaN,0.0,%,NaN,Centrum voor Grondonderzoek (C.V.G.),LABO
6,https://www.dov.vlaanderen.be/data/observatie/...,https://www.dov.vlaanderen.be/data/monster/196...,1968-01-01,NaN,NaN,Bodem_fysisch_textuur,Textuurfracties (textuurmeting),NaN,NaN,%,NaN,Centrum voor Grondonderzoek (C.V.G.),LABO
7,https://www.dov.vlaanderen.be/data/observatie/...,https://www.dov.vlaanderen.be/data/monster/196...,1968-01-01,NaN,NaN,Bodem_fysisch_textuur,Textuur - granulometrie - klasse bodemkarterin...,NaN,Z - Zand,NaN,NaN,Centrum voor Grondonderzoek (C.V.G.),LABO
8,https://www.dov.vlaanderen.be/data/observatie/...,https://www.dov.vlaanderen.be/data/monster/196...,1968-01-01,NaN,NaN,Bodem_fysisch_textuur,Textuur - handmatig - klassen bodemkartering (...,NaN,S - Lemig zand,-,NaN,Centrum voor Grondonderzoek (C.V.G.),LABO


### Buffer rond puntlocatie

Je kan ook zoeken op een cirkelvormige buffer rondom een puntlocatie:

In [11]:
from pydov.search.observatie import ObservatieSearch
from pydov.util.location import WithinDistance, Point

observatie_search = ObservatieSearch()

observatie_search.search(
    location=WithinDistance(
        Point(x=200000, y=205000, epsg=31370),
        distance=500)
)

[000/001] .


,pkey_observatie,pkey_parent,fenomeentijd,diepte_van_m,diepte_tot_m,parametergroep,parameter,detectieconditie,resultaat,eenheid,methode,uitvoerder,herkomst
0,https://www.dov.vlaanderen.be/data/observatie/...,https://www.dov.vlaanderen.be/data/monster/202...,2025-06-30,NaN,NaN,Zware metalen,Chroom (Cr),NaN,1.149,µg/l,Onbekend,Eurofins Analytico B.V.,LABO
1,https://www.dov.vlaanderen.be/data/observatie/...,https://www.dov.vlaanderen.be/data/monster/202...,2025-06-30,NaN,NaN,Anionen,Bicarbonaat (HCO3),NaN,5.5,mg/l,Onbekend,Eurofins Analytico B.V.,LABO
2,https://www.dov.vlaanderen.be/data/observatie/...,https://www.dov.vlaanderen.be/data/monster/202...,2025-06-30,NaN,NaN,Fysico-chemische parameters,Opgeloste zuurstof (O2),NaN,1.2,mg/l,Onbekend,Eurofins Analytico B.V.,VELD
3,https://www.dov.vlaanderen.be/data/observatie/...,https://www.dov.vlaanderen.be/data/monster/202...,2025-06-30,NaN,NaN,Kationen,Natrium (Na),NaN,13.0,mg/l,Onbekend,Eurofins Analytico B.V.,LABO
4,https://www.dov.vlaanderen.be/data/observatie/...,https://www.dov.vlaanderen.be/data/monster/202...,2025-06-30,NaN,NaN,Kationen,Kalium (K),NaN,10.7,mg/l,Onbekend,Eurofins Analytico B.V.,LABO
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3573,https://www.dov.vlaanderen.be/data/observatie/...,https://www.dov.vlaanderen.be/data/monster/202...,2022-11-17,NaN,NaN,Anionen,Sulfaat (SO4),NaN,49.8,mg/l,Onbekend,Eurofins Analytico B.V.,LABO
3574,https://www.dov.vlaanderen.be/data/observatie/...,https://www.dov.vlaanderen.be/data/monster/202...,2022-11-17,NaN,NaN,Anionen,Chloriden (Cl),NaN,32.7,mg/l,Onbekend,Eurofins Analytico B.V.,LABO
3575,https://www.dov.vlaanderen.be/data/observatie/...,https://www.dov.vlaanderen.be/data/monster/202...,2022-11-17,NaN,NaN,Zware metalen,Aluminium (Al),NaN,0.104,mg/l,Onbekend,Eurofins Analytico B.V.,LABO
3576,https://www.dov.vlaanderen.be/data/observatie/...,https://www.dov.vlaanderen.be/data/monster/202...,2022-11-17,NaN,NaN,Kationen,Kalium (K),NaN,22.0,mg/l,Onbekend,Eurofins Analytico B.V.,LABO


### GeoPandas geodataframe

Je kan ook geografisch zoeken op basis van een Geodataframe. Dit kan je gebruiken in een GeopandasFilter factory, tesamen met een locatiefilter.

Hieronder maken we eerst een geodataframe aan:

In [12]:
import geopandas as gpd

shapefile = "../../tests/data/util/location/polygon_multiple_31370.shp"

geodataframe = gpd.read_file(shapefile)
geodataframe["name"] = ["site 1", "site 2"]
geodataframe

,gml_id,geometry,name
0,polygon_multiple_31370.0,"POLYGON ((108636.15 194960.844, 109195.574 195...",site 1
1,polygon_multiple_31370.1,"POLYGON ((107485.786 196741.544, 108297.344 19...",site 2


Dit kunnen we nu gebruiken in een pydov zoekopdracht, bijvoorbeeld om boringen te vinden:

In [13]:
from pydov.search.boring import BoringSearch
from pydov.util.location import Within, GeopandasFilter

boring_search = BoringSearch()

boring_search.search(
    location=GeopandasFilter(geodataframe, Within)
)

[000/001] .
[000/018] cccccccccccccccccc


,pkey_boring,boornummer,x,y,mv_mtaw,start_boring_mtaw,gemeente,diepte_boring_van,diepte_boring_tot,datum_aanvang,uitvoerder,boorgatmeting,diepte_methode_van,diepte_methode_tot,boormethode
0,https://www.dov.vlaanderen.be/data/boring/2018...,B/4-104356,108025.00,196593.00,8.05,8.05,Gent,0.0,7.0,NaN,NaN,False,0.0,0.0,onbekend
1,https://www.dov.vlaanderen.be/data/boring/2019...,1718-B-180092,107947.29,196640.52,7.96,7.96,NaN,0.0,8.0,2019-03-11,Verhofste,False,0.0,8.0,spoelboring
2,https://www.dov.vlaanderen.be/data/boring/2020...,1718-B-190135,107991.00,196706.00,8.40,8.40,NaN,0.0,8.0,2020-05-15,Verhofste,False,0.0,8.0,spoelboring
3,https://www.dov.vlaanderen.be/data/boring/2022...,1407-B0863,107842.53,196371.08,7.38,7.38,Gent,0.0,4.0,2022-06-08,De Backer Putboringen,False,0.0,4.0,spoelboring
4,https://www.dov.vlaanderen.be/data/boring/2023...,1718-B220073,108144.95,196771.09,9.18,9.18,NaN,0.0,4.0,2023-04-13,Verhofste,False,0.0,4.0,spoelboring
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,https://www.dov.vlaanderen.be/data/boring/1893...,kb22d55e-B44,108900.00,194425.00,6.00,6.00,Destelbergen,0.0,21.0,1893-01-01,Behiels-(Lemmens)-Wetteren,False,0.0,21.0,onbekend
64,https://www.dov.vlaanderen.be/data/boring/1894...,kb22d55e-B102,107618.00,196709.00,7.50,7.50,Gent,0.0,0.0,1894-01-01,onbekend,False,0.0,0.0,onbekend
65,https://www.dov.vlaanderen.be/data/boring/1894...,kb22d55e-B103,107791.00,196516.00,7.50,7.50,Gent,0.0,0.0,1894-01-01,onbekend,False,0.0,0.0,onbekend
66,https://www.dov.vlaanderen.be/data/boring/1895...,kb22d55e-B400,109050.00,194990.00,7.00,7.00,Destelbergen,0.0,0.0,1895-01-01,onbekend,False,0.0,0.0,onbekend


We kunnen ook rechtstreeks een GIS bestand (bijvoorbeeld Shapefile) gebruiken met een GeometryFilter factory:

In [14]:
from pydov.search.boring import BoringSearch
from pydov.util.location import Within, GeometryFilter

boring_search = BoringSearch()

studiegebied = '../../tests/data/util/location/polygon_multiple_31370.shp'

boring_search.search(
    location=GeometryFilter(studiegebied, Within)
)

[000/001] .
[000/018] cccccccccccccccccc


,pkey_boring,boornummer,x,y,mv_mtaw,start_boring_mtaw,gemeente,diepte_boring_van,diepte_boring_tot,datum_aanvang,uitvoerder,boorgatmeting,diepte_methode_van,diepte_methode_tot,boormethode
0,https://www.dov.vlaanderen.be/data/boring/2018...,B/4-104356,108025.00,196593.00,8.05,8.05,Gent,0.0,7.0,NaN,NaN,False,0.0,0.0,onbekend
1,https://www.dov.vlaanderen.be/data/boring/2019...,1718-B-180092,107947.29,196640.52,7.96,7.96,NaN,0.0,8.0,2019-03-11,Verhofste,False,0.0,8.0,spoelboring
2,https://www.dov.vlaanderen.be/data/boring/2020...,1718-B-190135,107991.00,196706.00,8.40,8.40,NaN,0.0,8.0,2020-05-15,Verhofste,False,0.0,8.0,spoelboring
3,https://www.dov.vlaanderen.be/data/boring/2022...,1407-B0863,107842.53,196371.08,7.38,7.38,Gent,0.0,4.0,2022-06-08,De Backer Putboringen,False,0.0,4.0,spoelboring
4,https://www.dov.vlaanderen.be/data/boring/2023...,1718-B220073,108144.95,196771.09,9.18,9.18,NaN,0.0,4.0,2023-04-13,Verhofste,False,0.0,4.0,spoelboring
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,https://www.dov.vlaanderen.be/data/boring/1893...,kb22d55e-B44,108900.00,194425.00,6.00,6.00,Destelbergen,0.0,21.0,1893-01-01,Behiels-(Lemmens)-Wetteren,False,0.0,21.0,onbekend
64,https://www.dov.vlaanderen.be/data/boring/1894...,kb22d55e-B102,107618.00,196709.00,7.50,7.50,Gent,0.0,0.0,1894-01-01,onbekend,False,0.0,0.0,onbekend
65,https://www.dov.vlaanderen.be/data/boring/1894...,kb22d55e-B103,107791.00,196516.00,7.50,7.50,Gent,0.0,0.0,1894-01-01,onbekend,False,0.0,0.0,onbekend
66,https://www.dov.vlaanderen.be/data/boring/1895...,kb22d55e-B400,109050.00,194990.00,7.00,7.00,Destelbergen,0.0,0.0,1895-01-01,onbekend,False,0.0,0.0,onbekend


## Zoeken op attributen
> Meer info: https://pydov.readthedocs.io/en/stable/query_attribute.html

Naast zoeken op locatie, kan je ook zoeken naar objecten met bepaalde eigenschappen.

### Beschikbare zoekvelden

Om na te gaan welke attributen (velden) er beschikbaar zijn in een datatype, kan je de methode `get_fields()` gebruiken. Om specifiek de velden op te vragen waarop je kan zoeken gebruik je de volgende code:

In [15]:
from pydov.search.monster import MonsterSearch

monster_search = MonsterSearch()

monster_search.get_fields(query=True)

{'id': {'name': 'id', 'definition': 'Uniek referentienummer in de databank.', 'type': 'string', 'multivalue': False, 'notnull': False, 'query': True, 'cost': 1}, 'naam': {'name': 'naam', 'definition': 'Naam van het monster.', 'type': 'string', 'multivalue': False, 'notnull': False, 'query': True, 'cost': 1}, 'pkey_monster': {'name': 'pkey_monster', 'definition': 'Permanente URL die verwijst naar de gegevens van het monster op de website.', 'type': 'string', 'multivalue': False, 'notnull': False, 'query': True, 'cost': 1}, 'gekoppeld_aan': {'name': 'gekoppeld_aan', 'definition': 'Het ouder-object (parent) waaraan een monster gekoppeld is. Bijvoorbeeld een boring, een ander monster, ...', 'type': 'string', 'multivalue': False, 'notnull': False, 'query': True, 'cost': 1}, 'pkey_parents': {'name': 'pkey_parents', 'definition': 'Permanente URL die verwijst naar de gegevens van het gekoppeld ouder-object (parent) op de website.', 'type': 'string', 'multivalue': True, 'notnull': False, 'query': True, 'cost': 1}, 'materiaalklasse': {'name': 'materiaalklasse', 'definition': 'Aard van het materiaal waaruit het monster bestaat, ingedeeld volgens een classificatie bijvoorbeeld sediment, hard gesteente.', 'type': 'string', 'multivalue': False, 'notnull': False, 'query': True, 'cost': 1}, 'diepte_van_m': {'name': 'diepte_van_m', 'definition': 'Minimum diepte van het monster ten opzichte van het aanvangspeil van het gekoppeld ouder-object, in meter.', 'type': 'float', 'multivalue': False, 'notnull': False, 'query': True, 'cost': 1}, 'diepte_tot_m': {'name': 'diepte_tot_m', 'definition': 'Maximum diepte van het monster ten opzichte van het aanvangspeil van het gekoppeld ouder-object, in meter.', 'type': 'float', 'multivalue': False, 'notnull': False, 'query': True, 'cost': 1}, 'aantal_observaties': {'name': 'aantal_observaties', 'definition': 'Aantal observaties die beschikbaar zijn bij het monster.', 'type': 'integer', 'multivalue': False, 'notnull': False, 'query': True, 'cost': 1}, 'datum_monstername': {'name': 'datum_monstername', 'definition': 'Datum van monstername.', 'type': 'date', 'multivalue': False, 'notnull': False, 'query': True, 'cost': 1}, 'monstersamenstelling': {'name': 'monstersamenstelling', 'definition': 'Samenstelling van het monster: enkelvoudig monster of mengmonster.', 'type': 'string', 'multivalue': False, 'notnull': False, 'query': True, 'cost': 1}, 'monstertype': {'name': 'monstertype', 'definition': 'Type monster of bemonstering: geroerd, ongeroerd, vloeistof.', 'type': 'string', 'multivalue': False, 'notnull': False, 'query': True, 'cost': 1, 'codelist': <pydov.util.codelists.AbstractCodeList: <pydov.util.codelists.CodeListItem: code: geroerd, label: Geroerd, definition: Monstername waarbij de oorspronkelijke structuur en gelaagdheid van het materiaal niet bewaard wordt. Het resulterende monster laat het niet toe een gedetailleerde beschrijving of bepaalde analyses met betrekking tot de structuur of gelaagdheid  (bv. Bulkdensiteit, volumemassa, grondmechanische proeven, ....) uit te voeren.>, <pydov.util.codelists.CodeListItem: code: ongeroerd, label: Ongeroerd, definition: Monstername waarbij de oorspronkelijke structuur en gelaagdheid van het materiaal maximaal bewaard wordt.>, <pydov.util.codelists.CodeListItem: code: vloeistof, label: Vloeistof, definition: Het monster bestaat uit een van nature vloeibare stof (een stof die gemakkelijk vormveranderingen ondergaat, samendrukbaar is, maar zich verzet tegen deze volumeverandering). Het begrip gelaagdheid is niet relevant voor het karakteriseren van het materiaal waaruit het monster is genomen.>>}, 'bemonsteringsprocedure': {'name': 'bemonsteringsprocedure', 'definition': 'Een workflow, protocol, plan, algoritme of berekeningswijze waarin wordt gespecifieerd hoe de bemonstering moet worden uitgevoerd.', 'type': 'string', 'multivalue': False, 'notnull': False, 'query': True, 'cost': 1}, 'bemonsteringsinstrument': {'name': 'bemonsteringsinstrument', 'definition': 'H

### Attribuut gelijk aan

Zoeken op attribuutwaarden kan met de zoekoperatoren uit `owslib.fes2`: PropertyIsEqualTo, PropertyIsNotEqualTo, PropertyIsNull, PropertyIsLike, PropertyIsLessThan, PropertyIsLessThanOrEqualTo, PropertyIsGreaterThan, PropertyIsGreaterThanOrEqualTo, PropertyIsBetween.

Om bijvoorbeeld monsters op te vragen waarbij het monstertype ongeroerd is kan je volgende code gebruiken. De parameter `max_features` zorgt ervoor dat er maximaal dit aantal features teruggegeven worden, dat is handig om je query te testen op een subset van de data.

In [16]:
from pydov.search.monster import MonsterSearch

from owslib.fes2 import PropertyIsEqualTo

monster_search = MonsterSearch()

monster_search.search(
    query=PropertyIsEqualTo('monstertype', 'ongeroerd'),
    max_features=10
)

[000/001] .


,pkey_monster,naam,pkey_parents,materiaalklasse,datum_monstername,diepte_van_m,diepte_tot_m,monstertype,monstersamenstelling,bemonsteringsprocedure,bemonsteringsinstrument,bemonstering_door
0,https://www.dov.vlaanderen.be/data/monster/201...,0,(https://www.dov.vlaanderen.be/data/boring/198...,sediment,1980-01-01,288.80,288.80,ongeroerd,Enkelvoudig monster,NaN,"(buis,)",NaN
1,https://www.dov.vlaanderen.be/data/monster/201...,0,(https://www.dov.vlaanderen.be/data/boring/190...,hardgesteente,1901-01-01,541.00,542.20,ongeroerd,Enkelvoudig monster,NaN,"(kerboor,)",NaN
2,https://www.dov.vlaanderen.be/data/monster/201...,0.002,(https://www.dov.vlaanderen.be/data/boring/196...,sediment,1964-01-01,17.38,17.38,ongeroerd,Enkelvoudig monster,NaN,"(kerboor,)",NaN
3,https://www.dov.vlaanderen.be/data/monster/201...,1,(https://www.dov.vlaanderen.be/data/boring/199...,sediment,1995-01-01,102.25,102.25,ongeroerd,Enkelvoudig monster,NaN,"(kerboor,)",NaN
4,https://www.dov.vlaanderen.be/data/monster/201...,1,(https://www.dov.vlaanderen.be/data/boring/201...,sediment,1972-08-09,0.25,0.25,ongeroerd,Enkelvoudig monster,NaN,"(buis,)",NaN
5,https://www.dov.vlaanderen.be/data/monster/201...,1,(https://www.dov.vlaanderen.be/data/boring/197...,sediment,1972-01-01,0.20,0.20,ongeroerd,Enkelvoudig monster,NaN,"(kerboor,)",NaN
6,https://www.dov.vlaanderen.be/data/monster/201...,1,(https://www.dov.vlaanderen.be/data/boring/197...,sediment,1972-01-01,0.15,0.15,ongeroerd,Enkelvoudig monster,NaN,"(kerboor,)",NaN
7,https://www.dov.vlaanderen.be/data/monster/201...,1,(https://www.dov.vlaanderen.be/data/boring/197...,sediment,1972-01-01,0.20,0.20,ongeroerd,Enkelvoudig monster,NaN,"(kerboor,)",NaN
8,https://www.dov.vlaanderen.be/data/monster/201...,1,(https://www.dov.vlaanderen.be/data/boring/197...,sediment,1972-01-01,0.10,0.10,ongeroerd,Enkelvoudig monster,NaN,"(kerboor,)",NaN
9,https://www.dov.vlaanderen.be/data/monster/201...,1,(https://www.dov.vlaanderen.be/data/boring/197...,sediment,1972-01-01,0.80,0.80,ongeroerd,Enkelvoudig monster,NaN,"(kerboor,)",NaN


### Attribuut groter dan of gelijk aan

Om bijvoorbeeld alle monsters genomen sinds 1/1/2025 op te vragen gebruik je volgende code:

In [17]:
from pydov.search.monster import MonsterSearch

from owslib.fes2 import PropertyIsGreaterThanOrEqualTo

monster_search = MonsterSearch()

monster_search.search(
    query=PropertyIsGreaterThanOrEqualTo(
        'datum_monstername', '2025-01-01'),
    max_features=10
)

[000/001] .


,pkey_monster,naam,pkey_parents,materiaalklasse,datum_monstername,diepte_van_m,diepte_tot_m,monstertype,monstersamenstelling,bemonsteringsprocedure,bemonsteringsinstrument,bemonstering_door
0,https://www.dov.vlaanderen.be/data/monster/202...,000/00/2-F1/M2025,(https://www.dov.vlaanderen.be/data/filter/200...,grondwater,2025-04-22,NaN,3.0,vloeistof,Enkelvoudig monster,NaN,"(pomp,)",Eurofins Analytico B.V.
1,https://www.dov.vlaanderen.be/data/monster/202...,000/32/2-F1/M2025,(https://www.dov.vlaanderen.be/data/filter/200...,grondwater,2025-04-17,NaN,3.0,vloeistof,Enkelvoudig monster,NaN,"(pomp,)",Eurofins Analytico B.V.
2,https://www.dov.vlaanderen.be/data/monster/202...,000/32/2-F2/M2025,(https://www.dov.vlaanderen.be/data/filter/200...,grondwater,2025-04-17,NaN,6.5,vloeistof,Enkelvoudig monster,NaN,"(pomp,)",Eurofins Analytico B.V.
3,https://www.dov.vlaanderen.be/data/monster/202...,000/32/3-F1/M2025,(https://www.dov.vlaanderen.be/data/filter/200...,grondwater,2025-06-05,NaN,3.0,vloeistof,Enkelvoudig monster,NaN,"(pomp,)",Eurofins Analytico B.V.
4,https://www.dov.vlaanderen.be/data/monster/202...,000/32/3-F2/M2025,(https://www.dov.vlaanderen.be/data/filter/200...,grondwater,2025-06-05,NaN,6.0,vloeistof,Enkelvoudig monster,NaN,"(pomp,)",Eurofins Analytico B.V.
5,https://www.dov.vlaanderen.be/data/monster/202...,000/32/5-F1/M2025,(https://www.dov.vlaanderen.be/data/filter/200...,grondwater,2025-04-17,NaN,3.0,vloeistof,Enkelvoudig monster,NaN,"(pomp,)",Eurofins Analytico B.V.
6,https://www.dov.vlaanderen.be/data/monster/202...,000/32/5-F2/M2025,(https://www.dov.vlaanderen.be/data/filter/200...,grondwater,2025-04-17,NaN,5.9,vloeistof,Enkelvoudig monster,NaN,"(pomp,)",Eurofins Analytico B.V.
7,https://www.dov.vlaanderen.be/data/monster/202...,000/32/6-F1/M2025,(https://www.dov.vlaanderen.be/data/filter/200...,grondwater,2025-04-22,NaN,3.0,vloeistof,Enkelvoudig monster,NaN,"(pomp,)",Eurofins Analytico B.V.
8,https://www.dov.vlaanderen.be/data/monster/202...,000/32/6-F2/M2025,(https://www.dov.vlaanderen.be/data/filter/200...,grondwater,2025-04-22,NaN,7.2,vloeistof,Enkelvoudig monster,NaN,"(pomp,)",Eurofins Analytico B.V.
9,https://www.dov.vlaanderen.be/data/monster/202...,000/32/9-F1/M2025,(https://www.dov.vlaanderen.be/data/filter/200...,grondwater,2025-04-22,NaN,3.5,vloeistof,Enkelvoudig monster,NaN,"(pomp,)",Eurofins Analytico B.V.


### Attribuutfilters combineren

Het is ook mogelijk om verschillende filters (ook genest) te combineren met de And, Or, Not operatoren uit `owslib.fes2`.

Om de twee voorgaande zoekopdrachten te combineren gebruik je bijvoorbeeld:

In [18]:
from pydov.search.monster import MonsterSearch

from owslib.fes2 import (
    And, PropertyIsEqualTo, PropertyIsGreaterThanOrEqualTo)

monster_search = MonsterSearch()

monster_search.search(
    query=And([
        PropertyIsGreaterThanOrEqualTo('datum_monstername', '2025-01-01'),
        PropertyIsEqualTo('monstertype', 'ongeroerd')
    ])
)

[000/001] .


,pkey_monster,naam,pkey_parents,materiaalklasse,datum_monstername,diepte_van_m,diepte_tot_m,monstertype,monstersamenstelling,bemonsteringsprocedure,bemonsteringsinstrument,bemonstering_door
0,https://www.dov.vlaanderen.be/data/monster/202...,N001,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-02-05,4.00,4.50,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
1,https://www.dov.vlaanderen.be/data/monster/202...,N001,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-02-07,5.00,5.40,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
2,https://www.dov.vlaanderen.be/data/monster/202...,N001,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-01-21,2.50,3.00,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
3,https://www.dov.vlaanderen.be/data/monster/202...,N001,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-01-22,3.00,3.50,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
4,https://www.dov.vlaanderen.be/data/monster/202...,N001,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-02-28,6.50,7.00,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
...,...,...,...,...,...,...,...,...,...,...,...,...
233,https://www.dov.vlaanderen.be/data/monster/202...,N008,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-08-29,21.00,21.41,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
234,https://www.dov.vlaanderen.be/data/monster/202...,N008A,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-03-17,17.00,17.35,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
235,https://www.dov.vlaanderen.be/data/monster/202...,N008B,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-03-17,17.35,17.50,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
236,https://www.dov.vlaanderen.be/data/monster/202...,N009,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-03-13,17.00,17.50,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab


### Attribuutfilters combineren met locatie

Tenslotte kan je attribuutfilters ook combineren met een locatiefilter. Hierbij krijg je enkel resultaten die aan beide filters voldoen.

In [19]:
from pydov.search.monster import MonsterSearch

from owslib.fes2 import (
    And, PropertyIsEqualTo, PropertyIsGreaterThanOrEqualTo)

monster_search = MonsterSearch()

monster_search.search(
    query=And([
        PropertyIsGreaterThanOrEqualTo('datum_monstername', '2025-01-01'),
        PropertyIsEqualTo('monstertype', 'ongeroerd')
    ]),
    location=Within(Box(18000, 200000, 220000, 230000, epsg=31370))
)

[000/001] .


,pkey_monster,naam,pkey_parents,materiaalklasse,datum_monstername,diepte_van_m,diepte_tot_m,monstertype,monstersamenstelling,bemonsteringsprocedure,bemonsteringsinstrument,bemonstering_door
0,https://www.dov.vlaanderen.be/data/monster/202...,N001,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-02-05,4.0,4.50,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
1,https://www.dov.vlaanderen.be/data/monster/202...,N001,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-02-07,5.0,5.40,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
2,https://www.dov.vlaanderen.be/data/monster/202...,N001,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-02-28,6.5,7.00,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
3,https://www.dov.vlaanderen.be/data/monster/202...,N001,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-02-28,5.5,6.00,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
4,https://www.dov.vlaanderen.be/data/monster/202...,N001,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-03-04,4.5,5.00,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
...,...,...,...,...,...,...,...,...,...,...,...,...
91,https://www.dov.vlaanderen.be/data/monster/202...,N007,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-03-25,12.0,12.50,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
92,https://www.dov.vlaanderen.be/data/monster/202...,N007,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-03-14,13.5,14.00,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
93,https://www.dov.vlaanderen.be/data/monster/202...,N007,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-08-29,17.0,17.50,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
94,https://www.dov.vlaanderen.be/data/monster/202...,N008,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-03-14,16.5,16.90,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab


## Resultaatvelden selecteren

Als je minder (of: meer) velden nodig hebt dan er standaard aanwezig zijn in het resultaat, kan je de parameter `return_fields` gebruiken om deze velden aan te passen.

Je vraagt best enkel de velden op die je nodig hebt, dit zorgt ervoor dat de services niet onnodig belast worden en dat het resultaat ook sneller beschikbaar zal zijn.

In [20]:
from pydov.search.monster import MonsterSearch

monster_search = MonsterSearch()

monster_search.get_fields()

df_monsters = monster_search.search(
    max_features=10,
    return_fields=['pkey_monster', 'naam', 'diepte_van_m', 'diepte_tot_m']
)

df_monsters

[000/001] .


,pkey_monster,naam,diepte_van_m,diepte_tot_m
0,https://www.dov.vlaanderen.be/data/monster/201...,0,79.0,79.0
1,https://www.dov.vlaanderen.be/data/monster/201...,0,7.0,7.0
2,https://www.dov.vlaanderen.be/data/monster/201...,0,10.5,10.5
3,https://www.dov.vlaanderen.be/data/monster/201...,0,288.8,288.8
4,https://www.dov.vlaanderen.be/data/monster/201...,0,0.1,0.1
5,https://www.dov.vlaanderen.be/data/monster/201...,0,11.0,12.0
6,https://www.dov.vlaanderen.be/data/monster/201...,0,541.0,542.2
7,https://www.dov.vlaanderen.be/data/monster/201...,000/00/2-F1/M1501,NaN,3.0
8,https://www.dov.vlaanderen.be/data/monster/201...,000/00/2-F1/M1/C2014-1,NaN,3.0
9,https://www.dov.vlaanderen.be/data/monster/201...,000/00/2-F1/M2015-2,NaN,3.0


### Geometrie toevoegen

Standaard bevatten de resultaat dataframes enkel attribuutwaarden. Om ook de geometrie op te vragen kan je dit veld toevoegen aan de lijst met `return_fields`.

Om de naam van de geometrie kolom te bekomen kan je volgende opdracht gebruiken:

In [21]:
from pydov.search.monster import MonsterSearch

monster_search = MonsterSearch()

monster_search.get_fields(type='geometry')

{'geom': {'name': 'geom', 'definition': None, 'type': 'geometry', 'multivalue': False, 'notnull': False, 'query': False, 'cost': 1}}

Vervolgens kan je deze toevoegen als `GeometryReturnField`, waarbij je naast de naam ook het gewenst coördinatensysteem opgeeft:

In [22]:
from pydov.search.fields import GeometryReturnField

df_monsters = monster_search.search(
    max_features=10,
    return_fields=['pkey_monster', 'naam', GeometryReturnField('geom', epsg=31370)]
)

df_monsters

[000/001] .


,pkey_monster,naam,geom
0,https://www.dov.vlaanderen.be/data/monster/201...,0,POINT (236820 181175)
1,https://www.dov.vlaanderen.be/data/monster/201...,0,POINT (160032.9 175678.8)
2,https://www.dov.vlaanderen.be/data/monster/201...,0,POINT (153903 178252)
3,https://www.dov.vlaanderen.be/data/monster/201...,0,POINT (78776 226370)
4,https://www.dov.vlaanderen.be/data/monster/201...,0,POINT (214069.5 216345.4)
5,https://www.dov.vlaanderen.be/data/monster/201...,0,POINT (165215 161534)
6,https://www.dov.vlaanderen.be/data/monster/201...,0,POINT (234942 189926)
7,https://www.dov.vlaanderen.be/data/monster/201...,000/00/2-F1/M1501,POINT (28554.55 194470.43)
8,https://www.dov.vlaanderen.be/data/monster/201...,000/00/2-F1/M1/C2014-1,POINT (28554.55 194470.43)
9,https://www.dov.vlaanderen.be/data/monster/201...,000/00/2-F1/M2015-2,POINT (28554.55 194470.43)


Dit resultaat kan je eenvoudig omzetten naar een geodataframe:

In [24]:
gdf_monsters = gpd.GeoDataFrame(df_monsters, geometry='geom', crs='EPSG:31370')
gdf_monsters.explore()

## Datasets combineren

Naast de uitgebreide zoekmogelijkheden zit de kracht van pydov in de mogelijkheden om verschillende datasets te combineren. Zo kan je de resultaten van één dataset gebruiken om verder te zoeken en zo verschillende datasets linken aan elkaar.

Zo kan je bijvoorbeeld op zoek gaan naar recente ongeroerde monsters van materiaalklasse 'sediment' in je studiegebied:

In [34]:
from pydov.search.monster import MonsterSearch

from owslib.fes2 import (
    And, PropertyIsEqualTo, PropertyIsGreaterThanOrEqualTo)

monster_search = MonsterSearch()

df_monsters = monster_search.search(
    query=And([
        PropertyIsGreaterThanOrEqualTo('datum_monstername', '2025-01-01'),
        PropertyIsEqualTo('monstertype', 'ongeroerd'),
        PropertyIsEqualTo('materiaalklasse', 'sediment')
    ]),
    location=Within(Box(18000, 200000, 220000, 230000, epsg=31370)),
)

df_monsters

[000/001] .


,pkey_monster,naam,pkey_parents,materiaalklasse,datum_monstername,diepte_van_m,diepte_tot_m,monstertype,monstersamenstelling,bemonsteringsprocedure,bemonsteringsinstrument,bemonstering_door
0,https://www.dov.vlaanderen.be/data/monster/202...,N001,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-02-05,4.0,4.50,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
1,https://www.dov.vlaanderen.be/data/monster/202...,N001,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-02-07,5.0,5.40,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
2,https://www.dov.vlaanderen.be/data/monster/202...,N001,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-02-28,6.5,7.00,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
3,https://www.dov.vlaanderen.be/data/monster/202...,N001,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-02-28,5.5,6.00,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
4,https://www.dov.vlaanderen.be/data/monster/202...,N001,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-03-04,4.5,5.00,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
...,...,...,...,...,...,...,...,...,...,...,...,...
91,https://www.dov.vlaanderen.be/data/monster/202...,N007,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-03-25,12.0,12.50,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
92,https://www.dov.vlaanderen.be/data/monster/202...,N007,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-03-14,13.5,14.00,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
93,https://www.dov.vlaanderen.be/data/monster/202...,N007,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-08-29,17.0,17.50,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab
94,https://www.dov.vlaanderen.be/data/monster/202...,N008,(https://www.dov.vlaanderen.be/data/boring/202...,sediment,2025-03-14,16.5,16.90,ongeroerd,Enkelvoudig monster,NaN,"(steekbus,)",Geolab


En vervolgens de boringen opvragen waarvan deze monsters genomen zijn:

In [35]:
from pydov.search.boring import BoringSearch

from pydov.util.query import Join

boring_search = BoringSearch()

df_boringen = boring_search.search(
    query=Join(df_monsters, on='pkey_boring', using='pkey_parents'),
    return_fields=['pkey_boring', 'boornummer', 'x', 'y', 'start_boring_mtaw', 'diepte_boring_van', 'diepte_boring_tot']
)

df_boringen

[000/001] .


,pkey_boring,boornummer,x,y,start_boring_mtaw,diepte_boring_van,diepte_boring_tot
0,https://www.dov.vlaanderen.be/data/boring/2025...,1411-GEO-24/145-B2,171606.34,226099.24,29.47,0.0,17.0
1,https://www.dov.vlaanderen.be/data/boring/2025...,1411-GEO-24/170-B3,184946.81,224485.43,30.66,0.0,13.0
2,https://www.dov.vlaanderen.be/data/boring/2025...,1411-GEO-24/129-B1,190260.76,224986.24,30.80,0.0,11.0
3,https://www.dov.vlaanderen.be/data/boring/2025...,1411-GEO-24/172-B4,189057.08,224239.46,30.54,0.0,13.0
4,https://www.dov.vlaanderen.be/data/boring/2025...,1411-GEO-24/032-B2,184784.67,204405.32,24.97,0.0,13.0
5,https://www.dov.vlaanderen.be/data/boring/2025...,1411-GEO-24/048-B2,187330.14,203702.55,24.34,0.0,11.0
6,https://www.dov.vlaanderen.be/data/boring/2025...,1411-GEO-25/018-B2,138033.62,217213.13,11.02,0.0,15.0
7,https://www.dov.vlaanderen.be/data/boring/2025...,1411-GEO-25/016-B7,140207.22,220214.48,6.04,0.0,15.0
8,https://www.dov.vlaanderen.be/data/boring/2025...,1411-GEO-25/016-B40,139941.76,219840.94,10.64,0.0,15.0
9,https://www.dov.vlaanderen.be/data/boring/2025...,1411-GEO-25/016-B51,139930.52,220230.54,6.72,0.0,15.0
